In [ ]:
import torch
from diffusers import DiffusionPipeline

# 1. Configuração de Hardware
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
    variant = "fp16"
    print("Detectada GPU local!")

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
    variant = None
    print("Detectado Apple Silicon!")

else:
    device = "cpu"
    dtype = torch.float32
    variant = None
    print("Modo CPU Otimizado para SDXL.")

# 2. Carrega o SDXL Correto com travas de segurança de memória para CPU

print("Carregando Stable Diffusion XL Base 1.0...")

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    variant=variant if variant else None,
    low_cpu_mem_usage=True,          
    use_safetensors=True
).to(device)

# Libera memória do codificador de texto após o uso
if device == "cpu":
    pipe.enable_attention_slicing()  

print("SUCESSO: Modelo SDXL carregado com as dimensões corretas!")

In [ ]:
# 3. Injeta os pesos do LoRA apontando para a pasta 'treinamentos'
lora_weights_path = "../treinamento/pytorch_lora_weights.safetensors"

import os

if not os.path.exists(lora_weights_path):
    raise FileNotFoundError(
        f"Arquivo não encontrado em: '{lora_weights_path}':\n"
        "\n  1. Verifique se o nome da pasta é exatamente 'treinamento' (com ou sem maiúsculas) "
        "\n  2. Verifique se o arquivo pytorch_lora_weights.safetensors está na pasta."
    )

print(f"Injetando os pesos refinados do seu LoRA a partir de: {lora_weights_path}")

# Carrega os pesos passando o diretório correto e o nome do arquivo
pipe.load_lora_weights("../treinamento", weight_name="pytorch_lora_weights.safetensors")
print("SUCESSO: Estilo 'estilo_azulejaria' acoplado e pronto para uso!")


In [ ]:
import os
import sys
import random
import warnings

# Silencia avisos do Transformers
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

print("="*80)
print("GERADOR INTERATIVO AVANÇADO - ESTILO AZULEJARIA LUSO-BRASILEIRA")
print("*Exemplos: 'classic vintage car', 'majestic dragon', 'ceramic teapot'")
print("="*80)

# ============================================================================
# 1. Entrada do usuário
# ============================================================================
while True:
    user_input = input("Digite o objeto desejado (em inglês): ").strip().lower()
    if user_input:
        break

    print("OBS.: O objeto não pode ficar em branco!")

# ============================================================================
# 2. Tipo de composição
# ============================================================================
print("\nSELECIONE O ESTILO DE MONTAGEM DA CERÂMICA:")
print("  1. Painel Histórico ")
print("  2. Padrão Repetitivo ")

opcao = input("\nDigite sua escolha (1 ou 2): ").strip()

# ============================================================================
# 3. Paletas históricas
# ============================================================================

paletas_historicas = [
    "cobalt blue and white",
    "copper green and white",
    "yellow ochre and white",
    "terracotta and white"
]

cor_sorteada = random.choice(paletas_historicas)
print(f"\nPaleta selecionada: {cor_sorteada}")

# ============================================================================
# 4. Componentes compactos do prompt (Ajustados para Figura Avulsa)
# ============================================================================
trigger = "estilo_azulejaria"

materialidade = (
    "flat ceramic tiles, suttle grout lines, smooth glazed surface"
)

ornamentacao = (
    "decorative corner motifs on each tile, traditional cantoneiras"
)

qualidade = (
    "masterpiece, authentic antique look"
)

estilo_sorteado = None
# ============================================================================
# 5. Construção do prompt
# ============================================================================
if opcao == "2":

    print(f"\nGerando padrão repetitivo de Figura Avulsa para '{user_input}'.")

    prompt = (
        f"patchwork azulejo panel depicting a sharp detailed {user_input} alternating with plain white tiles, "
        f"{trigger}, {materialidade}, {cor_sorteada}, {qualidade}"
    )

    negative_prompt = (
        "deformed objects, fused tiles, broken grid, wavy lines, continuous landscape, photo, blurry, low quality"
    )
else:

    print(f"\nGerando painel histórico para '{user_input}'.")

    estilo_sorteado = random.choice(["continuo", "historico"])

    if estilo_sorteado == "continuo":
        # MONTAGEM 1: Objeto Monumental Contínuo (Estilo Tradicional)
        prompt = (
            f"A large {user_input} hand-painted on a continuous artwork painted across a ceramic tile mural, "
            f"subtle tile grout lines, elegant azulejo grid background, glossy glazed ceramic surface, "
            f"{trigger}, intricate blue and white patterns, {qualidade}"
        )

        negative_prompt = (
            "modern photo, abstract mess, blurry, low quality"
        )
    else:
        # MONTAGEM 2: Painel Tradicional Histórico (Estilo Moderno/Cenográfico)
        prompt = (
            f"historical azulejo panel depicting {user_input}, "
            f"{trigger}, {ornamentacao}, {materialidade}, {cor_sorteada}, {qualidade}"
        )

        negative_prompt = (

            "modern photo, abstract mess, blurry, low quality"
        )

print(f"  Tipo de painel: {estilo_sorteado}")

# ============================================================================
# 6. Exibição do prompt final
# ============================================================================
print("\nPROMPT FINAL:")
print(prompt)

# ============================================================================
# 7. Geração da imagem
# ============================================================================
print("\nRenderizando...")

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

# ============================================================================
# 8. Salvamento
# ============================================================================
output_dir = (
    "../resultados"
    if "google.colab" not in sys.modules
    else "/content/imagens_geradas_estilo"
)

os.makedirs(output_dir, exist_ok=True)

clean_name = "".join(
    [c for c in user_input if c.isalpha() or c.isspace()]
).strip().replace(" ", "_")

tipo_sufixo = "mural" if opcao != "2" else "padrao"

output_filename = os.path.join(
    output_dir,
    f"resultado_{clean_name}_{tipo_sufixo}.png"
)

image.save(output_filename)

print(f"\nImagem salva em: ")
print(output_filename)

display(image)